# A2 RNN's full model

# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [78]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell, Input, Concatenate, Lambda, Attention, GaussianNoise, Dropout
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose, RandomZoom, RandomTranslation, RandomRotation
from tensorflow.keras import Sequential

### Stuff

In [79]:
from scipy.ndimage import rotate
tf.random.set_seed(42)

# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [80]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


In [81]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


In [82]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


### More stuff

In [83]:
import tensorflow as tf
import numpy as np

def scheduled_mask_layer(x, training=None, prob=None):
    """
    Applies a Bernoulli mask to the input tensor for Scheduled Sampling.
    Reference: Bengio et al. (2015).
    """
    if training is None:
        training = tf.keras.backend.learning_phase()

    random_tensor = tf.random.uniform(tf.shape(x)[:2], minval=0, maxval=1, dtype=tf.float32)
    mask = tf.cast(random_tensor < prob, tf.float32)
    mask = tf.expand_dims(mask, axis=-1)


    return tf.where(tf.equal(training, True), x * mask, x)

class ScheduledMaskingLayer(tf.keras.layers.Layer):
    def __init__(self, prob_var, **kwargs):
        super(ScheduledMaskingLayer, self).__init__(**kwargs)
        self.prob_var = prob_var

    def call(self, inputs, training=None):
        return scheduled_mask_layer(inputs, training=training, prob=self.prob_var)

    def get_config(self):
        config = super().get_config()
        config.update({"prob_var": self.prob_var})
        return config

# 3. RE-INITIALIZE GLOBAL VARIABLE (Ensure it's a tf.Variable for Graph compatibility)
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32, name="sampling_prob_v5")

# Scheduled masking logic from your notebook
class ScheduledSamplingCallback(tf.keras.callbacks.Callback):
    def __init__(self, prob_var, decay_rate=0.05, min_prob=0.1):
        super().__init__()
        self.prob_var = prob_var      # The tf.Variable tracking the sampling probability
        self.decay_rate = decay_rate  # How much to reduce teacher forcing each epoch
        self.min_prob = min_prob      # The floor value (minimum teacher forcing retained)

    def on_epoch_end(self, epoch, logs=None):
        current_val = self.prob_var.numpy()
        new_val = max(self.min_prob, current_val - self.decay_rate)
        tf.keras.backend.set_value(self.prob_var, new_val)
        
        # 4. Log for monitoring convergence behavior
        print(f"\n --- End of Epoch {epoch + 1}: Teacher Forcing Ratio set to {new_val:.2f} ---")

In [84]:
def train_warmup(full_model, visual_encoder, x_train, y_train, val_data, epochs=5, learning_rate=1e-3, weight_decay=1.0e-4):
    visual_encoder.trainable = False
    
    optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
    full_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    
    sampling_cb = ScheduledSamplingCallback(sampling_prob, decay_rate=0.04, min_prob=0.5)
    
    history = full_model.fit(
        x=x_train, 
        y=y_train, 
        validation_data=val_data, 
        epochs=epochs, 
        batch_size=32,
        callbacks=[sampling_cb]
    )
    return history
import tensorflow as tf

def train_fine_tune(full_model, visual_encoder, x_train, y_train, val_data, epochs=50, learning_rate=1e-5, weight_decay=1.0e-2):

    visual_encoder.trainable = True
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=learning_rate, 
        weight_decay=weight_decay
    )
    
    full_model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy']
    )
    
    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    
    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10, 
        restore_best_weights=True,
        verbose=1
    )
    
    sampling_cb = ScheduledSamplingCallback(
        sampling_prob, 
        decay_rate=0.02, 
        min_prob=0.1
    )
    
    history = full_model.fit(
        x=x_train,
        y=y_train,
        validation_data=val_data,
        epochs=epochs,
        batch_size=32,
        callbacks=[early_stopper, lr_scheduler, sampling_cb]
    )
    
    return history

### Data

In [85]:
# Creating visual encoder training data, including ground-truth sequences used for teacher forcing.
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [86]:
# Calculator model data
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

In [87]:
# Full pipeline training data
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

## Full model

In [88]:
from tensorflow.keras.saving import load_model

custom_objects = {
    "scheduled_mask_layer": scheduled_mask_layer,
    "sampling_prob": sampling_prob
}

visual_encoder = load_model('visual_encoder.keras', custom_objects=custom_objects, safe_mode=False)
calculator = load_model('calculator.keras', custom_objects=custom_objects, safe_mode=False)

In [89]:
from tensorflow.keras import Model

def build_full_model(visual_encoder, calculator):
    img_input = Input(shape=(5, 28, 28, 1), name="img_input") 
    expr_tf_input = Input(shape=(6, 15), name="expr_tf_input") 
    ans_tf_input = Input(shape=(4, 15), name="ans_tf_input")   

    data_augmentation = Sequential([
    RandomRotation(0.05), # Rotate by ~18 degrees
    RandomTranslation(height_factor=0.15, width_factor=0.15), # Shift
    RandomZoom(0.15), # Zoom in/out
    ], name="spatial_augmentation")

    augmentation_layer = TimeDistributed(data_augmentation)(img_input)

    predicted_expression = visual_encoder([augmentation_layer, expr_tf_input])
    predicted_expression_dropout = Dropout(0.08)(predicted_expression)
    final_output = calculator([predicted_expression_dropout, ans_tf_input])

    return Model(inputs=[img_input, expr_tf_input, ans_tf_input], outputs=final_output)

model_full = build_full_model(visual_encoder, calculator)

In [90]:
import numpy as np

X_train_full = X_train[..., np.newaxis]
X_val_full = X_val[..., np.newaxis]

train_inputs = [X_train_full, y_train_in_pt, y_train_in]
val_inputs = [X_val_full, y_val_in_pt, y_val_in]

train_targets = y_train_target
val_targets = y_val_target

In [91]:
sampling_prob = tf.Variable(1.0, trainable=False, dtype=tf.float32)

def scheduled_mask_layer(x, training=None):
    if training:

        mask = tf.cast(tf.random.uniform(tf.shape(x)[:-1]) < sampling_prob, dtype=tf.float32)
        return x * tf.expand_dims(mask, -1)
    return x

# Something went wrong with a lambda layer. I asked gemini to help me fix it when building the full model. Sorry!
def patch_model_lambdas(model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Lambda):
            # We point the Lambda's function to our new local version
            layer.function = scheduled_mask_layer
    return model

# Apply the patch to your loaded models
visual_encoder = patch_model_lambdas(visual_encoder)
calculator = patch_model_lambdas(calculator)



In [92]:
model_full.summary(expand_nested=True)

Model: "functional_26"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_input           │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_5  │ (None, 5, 28, 28, │          0 │ img_input[0][0]   │
│ (TimeDistributed)   │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expr_tf_input       │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pretraining_model   │ (None, 6, 15)     │  1,254,991 │ time_distributed… │
│ (Functional)        │                   │            │ expr_tf_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ sequence       │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ encoder_model  │ [(None, 5, 7, 7,  │    906,560 │ -                 │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └ input_layer │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed    │ 1)                │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_1  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_2  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_3  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_4  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_5  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_7  │ 64)               │            │                 

 Total params: 1,951,582 (7.44 MB)

 Trainable params: 1,951,390 (7.44 MB)

 Non-trainable params: 192 (768.00 B)

In [93]:
# Now rebuild the full model
model_full = build_full_model(visual_encoder, calculator)

def run_full_pipeline_training(model_full, visual_encoder):

    visual_encoder.trainable = False
    
    print("Starting Stage 1: Warmup (Visual Encoder Frozen)...")
    history_warmup = train_warmup(
        full_model=model_full,
        visual_encoder=visual_encoder,
        x_train=train_inputs,
        y_train=train_targets,
        val_data=(val_inputs, val_targets),
        learning_rate=4.0e-4,
        weight_decay=1.0e-1
    )

    visual_encoder.trainable = True
    
    print("\nStarting Stage 2: Fine-Tuning (End-to-End)...")
    history_fine_tune = train_fine_tune(
        full_model=model_full,
        visual_encoder=visual_encoder,
        x_train=train_inputs,
        y_train=train_targets,
        val_data=(val_inputs, val_targets),
        learning_rate=1.0e-5,
        weight_decay=1.0e-1
    )
    
    return history_warmup, history_fine_tune

warmup_h, finetune_h = run_full_pipeline_training(model_full, visual_encoder)

Starting Stage 1: Warmup (Visual Encoder Frozen)...
Epoch 1/5
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6347 - loss: 1.2452
 --- End of Epoch 1: Teacher Forcing Ratio set to 0.96 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.6782 - loss: 1.0657 - val_accuracy: 0.7580 - val_loss: 0.8263
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7323 - loss: 0.8948
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.92 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.7382 - loss: 0.8750 - val_accuracy: 0.7776 - val_loss: 0.7590
Epoch 3/5
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7534 - loss: 0.8298
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.88 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.7544 - loss: 0.8247 - val_accuracy: 0.7910 - val_loss: 0.7183
Epoch 4/5
498/500 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7630 - loss: 0.7880
 --- End of Epoch 4: Teacher Forcing Ratio set to 0.84 ---
500/500 ━━━

E0000 00:00:1767900972.457869 3495646 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_136/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_136/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_952/gradient_tape/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - categorical_accuracy: 0.7800 - loss: 0.7264
 --- End of Epoch 1: Teacher Forcing Ratio set to 0.78 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 58ms/step - categorical_accuracy: 0.7853 - loss: 0.7109 - val_categorical_accuracy: 0.8879 - val_loss: 0.4494 - learning_rate: 1.0000e-05
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - categorical_accuracy: 0.7907 - loss: 0.6963
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.76 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - categorical_accuracy: 0.7933 - loss: 0.6899 - val_categorical_accuracy: 0.8944 - val_loss: 0.4245 - learning_rate: 1.0000e-05
Epoch 3/50
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - categorical_accuracy: 0.7971 - loss: 0.6800
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.74 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - categorical_accuracy: 0.7973 - loss: 0.6774 - val_categorical_accuracy: 0.8851 - val_loss: 0.4487 - learning_rate: 1.0000e-05
Epoch 4/50
500/500 ━━━━━

In [94]:
history_fine_tune = train_fine_tune(
        full_model=model_full,
        visual_encoder=visual_encoder,
        x_train=train_inputs,
        y_train=train_targets,
        val_data=(val_inputs, val_targets)
    )

Epoch 1/50


E0000 00:00:1767901316.610497 3495646 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_136/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/convolution_6' -> 'StatefulPartitionedCall/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_136/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_6', 'StatefulPartitionedCall/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_136/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_136/functional_28_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCal

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - categorical_accuracy: 0.7992 - loss: 0.6743
 --- End of Epoch 1: Teacher Forcing Ratio set to 0.54 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - categorical_accuracy: 0.8005 - loss: 0.6712 - val_categorical_accuracy: 0.8723 - val_loss: 0.4815 - learning_rate: 1.0000e-05
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - categorical_accuracy: 0.7983 - loss: 0.6756
 --- End of Epoch 2: Teacher Forcing Ratio set to 0.52 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 27s 54ms/step - categorical_accuracy: 0.8008 - loss: 0.6663 - val_categorical_accuracy: 0.8788 - val_loss: 0.4653 - learning_rate: 1.0000e-05
Epoch 3/50
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - categorical_accuracy: 0.8001 - loss: 0.6661
 --- End of Epoch 3: Teacher Forcing Ratio set to 0.50 ---
500/500 ━━━━━━━━━━━━━━━━━━━━ 27s 54ms/step - categorical_accuracy: 0.8009 - loss: 0.6659 - val_categorical_accuracy: 0.8779 - val_loss: 0.4626 - learning_rate: 1.0000e-05
Epoch 4/50
500/500 ━━━━━

# Inference

In [95]:
import numpy as np

def run_full_inference(visual_encoder, calculator, X_img_test, char_to_index):

    # Inverse mapping for decoding
    index_to_char = {v: k for k, v in char_to_index.items()}
    num_samples = X_img_test.shape[0]
    
    # 1. Prepare Image Input (Ensuring 5D shape [N, 5, 28, 28, 1])
    X_test_expanded = X_img_test[..., np.newaxis]
    
    print(f"Starting inference on {num_samples} samples...")
    

    dummy_expr = np.zeros((num_samples, 6, 15)) 

    pred_expressions_dist = visual_encoder.predict([X_test_expanded, dummy_expr], verbose=0)
    

    dummy_ans = np.zeros((num_samples, 4, 15))
    pred_answers_dist = calculator.predict([pred_expressions_dist, dummy_ans], verbose=0)
    

    results = []
    for i in range(num_samples):
        # Convert probability distributions to character sequences via Argmax [4]
        expr_indices = np.argmax(pred_expressions_dist[i], axis=-1)
        ans_indices = np.argmax(pred_answers_dist[i], axis=-1)
        
        expr_str = "".join([index_to_char[idx] for idx in expr_indices])
        ans_str = "".join([index_to_char[idx] for idx in ans_indices])
        
        results.append({
            "expression": expr_str.replace('<PAD>', '').replace('<EOS>', ''),
            "answer": ans_str.replace('<PAD>', '').replace('<EOS>', '')
        })
        
    return results

# Usage:
# results = run_full_inference(visual_encoder, calculator, X_test, char_to_index)

In [98]:
import numpy as np

def calculate_full_model_metrics(results, y_test_target, char_to_index):


    index_to_char = {v: k for k, v in char_to_index.items()}
    num_samples = len(results)
    
    total_tokens = 0
    correct_tokens = 0
    correct_math_results = 0
    
    # 1. Decode Ground Truth Answers
    y_true_indices = np.argmax(y_test_target, axis=-1)
    
    for i in range(num_samples):
        # Clean Ground Truth string
        true_ans_str = "".join([index_to_char[idx] for idx in y_true_indices[i]])
        true_ans_clean = true_ans_str.replace('<PAD>', '').replace('<EOS>', '')
        
        # Predicted Answer from inference loop
        pred_ans_clean = results[i]['answer']
        
        # --- A. Math Accuracy (Exact Match) ---
        # Checks if the entire sequence is identical to the target [2]
        if pred_ans_clean == true_ans_clean:
            correct_math_results += 1
            
        # --- B. Token Accuracy (Character Level) ---
        # We compare character by character up to the length of the shorter string [3]
        # (Alternatively, you can pad to fixed length for strict comparison)
        min_len = min(len(true_ans_clean), len(pred_ans_clean))
        max_len = max(len(true_ans_clean), len(pred_ans_clean))
        
        for char_idx in range(min_len):
            if true_ans_clean[char_idx] == pred_ans_clean[char_idx]:
                correct_tokens += 1
        
        total_tokens += max_len
    
    # 2. Compute Percentages    
    math_accuracy = (correct_math_results / num_samples) * 100
    token_accuracy = (correct_tokens / total_tokens) * 100
    
    print(f"--- Full Pipeline Evaluation ---")
    print(f"Math Accuracy (Exact Sequence Match): {math_accuracy:.2f}%")
    print(f"Token Accuracy (Character Level):     {token_accuracy:.2f}%")
    
    return math_accuracy, token_accuracy

# Usage:
# results = run_full_inference(visual_encoder, calculator, X_test, char_to_index)
# math_acc, token_acc = calculate_full_model_metrics(results, y_test_target, char_to_index)

In [99]:
results = run_full_inference(visual_encoder, calculator, X_test, indices)
math_acc, token_acc = calculate_full_model_metrics(results, y_test_target, indices)

Starting inference on 2000 samples...


--- Full Pipeline Evaluation ---
Math Accuracy (Exact Sequence Match): 22.40%
Token Accuracy (Character Level):     87.02%
